## Callback Handler e Context Manager

### Esempio: conteggio token

In [1]:
from langchain.chat_models import init_chat_model
import os
from langchain_core.callbacks import UsageMetadataCallbackHandler

model_1 = init_chat_model(model="openai:gpt-4o-mini")
model_2 = init_chat_model(model="openai:gpt-5-mini")

query = "Ciao, sono Enzo e mi piace l'arancione"

callback = UsageMetadataCallbackHandler()

# Un Callback Handler, in LangChain, è un oggetto che “si aggancia” (“hook”)
# ai vari eventi che accadono durante l’esecuzione di un modello, di una
# catena (chain), di un agente, ecc.
# L’obiettivo è intercettare azioni come “il modello sta per partire”,
# “il modello ha prodotto una risposta”, “c’è stato un errore”, ecc.,
# così da poter:
# - registrare metriche (es. token usati, tempo, costi)
# - tracciare i passaggi eseguiti
# - loggare input/output
# - effettuare streaming dei risultati o monitoraggio
# - ...(qualsiasi operazione necessitiamo)

# lista callback:
# https://reference.langchain.com/python/langchain_core/callbacks/

result_1 = model_1.invoke(query, config={"callbacks": [callback]})
result_2 = model_2.invoke(query, config={"callbacks": [callback]})

callback.usage_metadata

{'gpt-4o-mini-2024-07-18': {'input_tokens': 20,
  'output_tokens': 41,
  'total_tokens': 61,
  'input_token_details': {'audio': 0, 'cache_read': 0},
  'output_token_details': {'audio': 0, 'reasoning': 0}},
 'gpt-5-mini-2025-08-07': {'input_tokens': 19,
  'output_tokens': 564,
  'total_tokens': 583,
  'input_token_details': {'audio': 0, 'cache_read': 0},
  'output_token_details': {'audio': 0, 'reasoning': 384}}}

In [2]:
# tramite context manager

from langchain_core.callbacks import get_usage_metadata_callback

model_1 = init_chat_model(model="openai:gpt-4o-mini")
model_2 = init_chat_model(model="openai:gpt-5-mini")

with get_usage_metadata_callback() as cb:
    # La funzione get_usage_metadata_callback() restituisce un generatore/context-manager
    # che fornisce un handler interno e lo registra come callback per ogni chiamata al modello
    # nel blocco with. Alla fine, il campo cb.usage_metadata tiene traccia aggregata dell’uso
    # di token per tutti i modelli invocati nel blocco.
    # Vantaggi:
    # - Non devi passare manualmente il callback a ciascuna invoke.
    # - Raccoglie automaticamente tutti i modelli/chain eseguiti all’interno del blocco with.
    # - È più pulito quando hai più chiamate da tracciare in sequenza, senza ripetere configurazione.
    # - Alla uscita dal blocco viene garantita la “chiusura” del contesto (es. eventuali risorse
    #   liberate, handler deregistrato).
    #
    # È sicuramente più utile per comportamenti globali, non specifici di un singolo modello
    model_1.invoke(query)
    model_2.invoke(query)

print(cb.usage_metadata)

{'gpt-4o-mini-2024-07-18': {'input_tokens': 20, 'output_tokens': 37, 'total_tokens': 57, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, 'gpt-5-mini-2025-08-07': {'input_tokens': 19, 'output_tokens': 368, 'total_tokens': 387, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 256}}}
